In [1]:
import torch
from torch import nn
from torch import optim
from model import Glove
from dataset import Wiki103Dataset
from model import SentimentAnalysisLSTM
from trainer import Trainer

In [2]:
device = 'cuda'
hidden_dim = 512

In [3]:
glove = Glove()
model = SentimentAnalysisLSTM(glove.vocab_size, glove.embed_dim,hidden_dim,dtype=torch.float32)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())
embedding = nn.Embedding.from_pretrained( 
    glove.weight_matrix,
    freeze=True
).to(torch.float16).to(device)

Quick-loading Glove from cache...


General LM Pretraining

In [1]:
dataset = Wiki103Dataset()
trainer = Trainer(model,glove, embedding, optimizer, loss_fn, dataset)
trainer.train(batch_size=1,epochs=4,resume_file="ckpt_epoch_3_batch_137600.pth")

NameError: name 'Wiki103Dataset' is not defined

Task LM Training

In [5]:
from dataset import ProcessedEssayDataset
from trainer import Trainer

In [6]:
finetune_dataset = ProcessedEssayDataset().text
split = int(len(finetune_dataset) * 0.8) 
finetune_trainer = Trainer(model, glove, embedding, optimizer,loss_fn, finetune_dataset[:split],save_dir="TaskLMTraining")
finetune_trainer.train(batch_size=1,resume_file="ckpt_epoch_3_batch_137600.pth")

FileNotFoundError: [Errno 2] No such file or directory: 'processed_essay.csv'

Classification Task Training

In [4]:
from trainer import ClassifierTrainer
from dataset.dataset import ProcessedEssayDataset
from model import SimpleClassifier
from trainer.utils import load_model_from_trainer_checkpoint
import copy
import os
finetune_dataset = ProcessedEssayDataset(filename="dataset/processed_essay.csv")
split = int(len(finetune_dataset) *0.8)
load_model_from_trainer_checkpoint(model,"checkpoints/TaskLMTraining/ckpt_epoch_5_batch_1000.pth")

In [9]:
def train(lstm_instance,dataset,save_dir, epoch, index,existing_classifier = None):
    if not existing_classifier:
        classifier = SimpleClassifier(input_dim=hidden_dim, num_labels=2).to(device)
    else:
        classifier = existing_classifier
    trainer = ClassifierTrainer(lstm_instance, classifier, glove, embedding, device, trait_idx=0)
    os.makedirs(save_dir, exist_ok=True)
    # Training Loop
    for epoch in range(epoch):
        avg_loss = trainer.train_epoch(dataset[:split])
        print(f"Epoch {epoch} Avg Loss: {avg_loss:.4f}")
        trainer.save_checkpoint(f"{save_dir}/epoch{epoch}.pth")
    return lstm_instance, classifier

In [10]:
models = []
for i in range(5):
    lstm_instance = copy.deepcopy(models[i]['lstm']).to(device)
    lstm_instance, classifier = train(lstm_instance, finetune_dataset, f"checkpoints/TaskTraining/{finetune_dataset.labels[i+1]}",existing_classifier=models[i]['classifier'],epoch=7,index= i)
    lstm_instance.to('cpu')
    classifier.to('cpu')
    models[i] = {'lstm' : lstm_instance, 'classifier' : classifier}

IndexError: list index out of range

Evaluate

In [8]:
from model.pipeline import TraitEvaluator
for i in range(5):
    trait_name = finetune_dataset.labels[i+1]
    evaluator = TraitEvaluator(models[i]['lstm'].to(device),models[i]['classifier'].to(device) , glove, embedding, device, trait_idx=i)
    print(trait_name)
    evaluator.evaluate(finetune_dataset,sample_range=(split,len(finetune_dataset)))

cEXT

--- Confusion Matrix (Trait Index: 0) ---
                | Predicted: 0 | Predicted: 1
      Actual: 0 |     116      |     121     
      Actual: 1 |     119      |     138     
---------------------------------------------
Accuracy:  51.42%
Precision: 53.28%
Recall:    53.70%
F1-score:  53.49%

cNEU

--- Confusion Matrix (Trait Index: 1) ---
                | Predicted: 0 | Predicted: 1
      Actual: 0 |     157      |      99     
      Actual: 1 |     126      |     112     
---------------------------------------------
Accuracy:  54.45%
Precision: 53.08%
Recall:    47.06%
F1-score:  49.89%

cAGR

--- Confusion Matrix (Trait Index: 2) ---
                | Predicted: 0 | Predicted: 1
      Actual: 0 |     156      |      81     
      Actual: 1 |     161      |      96     
---------------------------------------------
Accuracy:  51.01%
Precision: 54.24%
Recall:    37.35%
F1-score:  44.24%

cCON

--- Confusion Matrix (Trait Index: 3) ---
                | Predicted: 0 | Pred

Load from File

In [7]:
from model.pipeline import TraitEvaluator
for i in range(5):
    trait_name = finetune_dataset.labels[i+1]
    classifier = SimpleClassifier(input_dim=hidden_dim, num_labels=2).to(device)
    evaluator = TraitEvaluator(models[i]['lstm'].to(device), classifier, glove, embedding, device, trait_idx=i)
    print(trait_name)
    evaluator.evaluate(finetune_dataset,sample_range=(split,len(finetune_dataset)))

cEXT

--- Confusion Matrix (Trait Index: 0) ---
                | Predicted: 0 | Predicted: 1
      Actual: 0 |     237      |      0      
      Actual: 1 |     256      |      1      
---------------------------------------------
Accuracy:  48.18%
Precision: 100.00%
Recall:    0.39%
F1-score:  0.78%

cNEU

--- Confusion Matrix (Trait Index: 1) ---
                | Predicted: 0 | Predicted: 1
      Actual: 0 |     108      |     148     
      Actual: 1 |      93      |     145     
---------------------------------------------
Accuracy:  51.21%
Precision: 49.49%
Recall:    60.92%
F1-score:  54.61%

cAGR

--- Confusion Matrix (Trait Index: 2) ---
                | Predicted: 0 | Predicted: 1
      Actual: 0 |     202      |      35     
      Actual: 1 |     207      |      50     
---------------------------------------------
Accuracy:  51.01%
Precision: 58.82%
Recall:    19.46%
F1-score:  29.24%

cCON

--- Confusion Matrix (Trait Index: 3) ---
                | Predicted: 0 | Predi